# Q1

In [35]:
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 

In [36]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

--2026-10-12 21:18:32--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.2’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.002s  

2026-10-12 21:18:32 (151 MB/s) - ‘course_lead_scoring_2026.csv.2’ saved [278746/278746]



In [37]:
!wget $data -O course_lead_scoring_2026.csv

--2026-10-12 21:18:32--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.002s  

2026-10-12 21:18:32 (120 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]



In [38]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()
df.isna().sum().sort_values(ascending=False)

annual_income               369
industry                    240
location                    208
employment_status           194
lead_source                 148
lead_score                   35
number_of_courses_viewed      0
interaction_count             0
converted                     0
dtype: int64

In [39]:
df.isna().sum().sort_values(ascending=False)

annual_income               369
industry                    240
location                    208
employment_status           194
lead_source                 148
lead_score                   35
number_of_courses_viewed      0
interaction_count             0
converted                     0
dtype: int64

In [40]:
target = "converted"

y = df[target].copy()
X = df.drop(columns=[target]).copy()

# Identificar tipos de columnas
categorical = X.select_dtypes(include=["object", "category", "string"]).columns
numerical = X.select_dtypes(include="number").columns

# Rellenar faltantes en features categóricas y numéricas
for col in categorical:
    if isinstance(X[col].dtype, pd.CategoricalDtype):
        X[col] = X[col].cat.add_categories(["NA"]).fillna("NA")
    else:
        X[col] = X[col].fillna("NA")

X[numerical] = X[numerical].fillna(0.0)

# El dataframe para el modelo contiene solo features, no el target
df = X.copy()

# Comprobar que ya no queden faltantes en las features
df.isna().sum().sum()

np.int64(0)

In [41]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score'],
      dtype='str')

In [42]:
X["industry"].value_counts().head()

industry
technology    1173
retail         925
healthcare     840
finance        720
education      639
Name: count, dtype: int64

In [43]:
X["industry"].mode().iat[0]

'technology'

# Q2

In [44]:
# Excluye la columna objetivo; la correlación es entre features
numeric_features = X.select_dtypes(include="number")
corr = numeric_features.corr()

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

results = sorted(
    [(a, b, corr.loc[a, b]) for a, b in pairs],
    key=lambda item: abs(item[2]),
    reverse=True,
)

for a, b, value in results:
    print(f"{a} y {b}: {value:.3f}")

interaction_count y lead_score: 0.916
number_of_courses_viewed y lead_score: 0.757
number_of_courses_viewed y interaction_count: 0.722
annual_income y interaction_count: 0.123


In [45]:
print(df.columns.tolist())

['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [46]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_full_train = y.loc[df_full_train.index]
y_test = y.loc[df_test.index]
y_train = y.loc[df_train.index]
y_val = y.loc[df_val.index]

In [47]:
print(df.columns.tolist())

['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [48]:
corr = df.select_dtypes(include="number").corr()
display(corr)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [49]:
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

results = sorted(
    [(a, b, corr.loc[a, b]) for a, b in pairs],
    key=lambda item: abs(item[2]),
    reverse=True,
)

for a, b, value in results:
    print(f"{a} y {b}: {value:.3f}")

interaction_count y lead_score: 0.916
number_of_courses_viewed y lead_score: 0.757
number_of_courses_viewed y interaction_count: 0.722
annual_income y interaction_count: 0.123


In [50]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# Q3

In [51]:
from sklearn.metrics import mutual_info_score

categorical_features = [
    "industry",
    "location",
    "lead_source",
    "employment_status",
]

mi_scores = {
    feature: round(mutual_info_score(y_train, df_train[feature]), 2)
    for feature in categorical_features
}

sorted(mi_scores.items(), key=lambda item: item[1], reverse=True)

[('lead_source', 0.03),
 ('employment_status', 0.02),
 ('industry', 0.0),
 ('location', 0.0)]

# Q4

In [53]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(df_train.to_dict(orient="records")).toarray()
X_val = dv.transform(df_val.to_dict(orient="records")).toarray()

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000,
    random_state=42,
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)
round(accuracy_score(y_val, y_pred), 2)

0.65

Q5

In [54]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

features_to_test = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count",
]

def validation_accuracy(features):
    dv = DictVectorizer()
    X_train = dv.fit_transform(
        df_train[features].to_dict(orient="records")
    ).toarray()
    X_val = dv.transform(
        df_val[features].to_dict(orient="records")
    ).toarray()

    model = LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=1000,
        random_state=42,
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    return accuracy_score(y_val, predictions)

all_features = df_train.columns.tolist()
baseline = validation_accuracy(all_features)

for feature in features_to_test:
    accuracy_without = validation_accuracy(
        [column for column in all_features if column != feature]
    )
    difference = baseline - accuracy_without
    print(
        f"{feature}: accuracy sin ella={accuracy_without:.6f}, "
        f"diferencia={difference:.6f}"
    )

lead_source: accuracy sin ella=0.642000, diferencia=0.003000
number_of_courses_viewed: accuracy sin ella=0.643000, diferencia=0.002000
interaction_count: accuracy sin ella=0.601000, diferencia=0.044000
